# Cross-lingual Mu-SHROOM evaluation (`es`, `hi`, `fi`)

This notebook is now **English-transfer only** for a fast proposal-aligned run on:
- `es` (closer to English)
- `hi` (different script/typology)
- `fi` (different morphology)

It does:
1. Evaluate the English fine-tuned model on each target language.
2. Score predictions with the official Mu-SHROOM scorer.
3. Output per-language IoU/Correlation tables for direct comparison.


In [ ]:
# Install dependencies (Colab-safe)
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn

import os
import json
import subprocess
import importlib.util
from typing import List, Tuple

import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Config (EN-transfer only)
MODEL_NAME = "xlm-roberta-large"
TARGET_LANGS = ["es", "hi", "fi"]

# English model path (from your existing Fine_Tuning notebook run)
EN_MODEL_DIR = "/content/xlmr-mu-shroom-en/best"
MAX_LENGTH = 512

# Optimized official-scoring params from EN tuning
BEST_THRESHOLD = 0.60
BEST_TEMPERATURE = 0.80
BEST_MIN_SPAN_LEN = 1
BEST_MERGE_GAP = 2

OUT_ROOT = "/content/mushroom_multilingual"
os.makedirs(OUT_ROOT, exist_ok=True)

print("Target languages:", TARGET_LANGS)
print("Mode: EN-transfer only")

In [ ]:
# Official scorer setup + shared helpers

def clone_if_missing(url: str, path: str):
    if not os.path.isdir(path):
        subprocess.check_call(["git", "clone", "--depth", "1", url, path])


def load_scorer_module(kit_dir: str):
    path = os.path.join(kit_dir, "scorer.py")
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


def write_jsonl(path: str, rows):
    with open(path, "w", encoding="utf-8") as w:
        for r in rows:
            w.write(json.dumps(r, ensure_ascii=False) + "\n")


def score_predictions(scorer_mod, ref_jsonl: str, pred_jsonl: str, scores_txt: str) -> Tuple[float, float]:
    ref_list = scorer_mod.load_jsonl_file_to_records(ref_jsonl, is_ref=True)
    pred_list = scorer_mod.load_jsonl_file_to_records(pred_jsonl, is_ref=False)
    ious, cors = scorer_mod.main(ref_list, pred_list, scores_txt)
    return float(ious.mean()), float(cors.mean())


def merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    if not spans:
        return []
    spans = sorted([[int(s), int(e)] for s, e in spans if e > s], key=lambda x: (x[0], x[1]))
    out = [spans[0]]
    for s, e in spans[1:]:
        ps, pe = out[-1]
        if s <= pe + gap:
            out[-1][1] = max(pe, e)
        else:
            out.append([s, e])
    return out


def predict_with_params(model_dir: str, ref_jsonl: str, out_jsonl: str):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    records = []
    with open(ref_jsonl, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    out_rows = []
    with torch.no_grad():
        for rec in records:
            enc = tokenizer(
                rec["model_output_text"],
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            )
            offsets = enc.pop("offset_mapping")[0]
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits[0]
            probs = F.softmax(logits / max(BEST_TEMPERATURE, 1e-6), dim=-1)

            soft_labels, hard_spans = [], []
            for j, off in enumerate(offsets):
                s, e = off[0].item(), off[1].item()
                if s >= e:
                    continue
                if probs.shape[-1] > 2:
                    p_hall = float((probs[j, 1] + probs[j, 2]).item())
                elif probs.shape[-1] == 2:
                    p_hall = float(probs[j, 1].item())
                else:
                    p_hall = 0.0
                soft_labels.append({"start": s, "end": e, "prob": p_hall})
                if p_hall >= BEST_THRESHOLD:
                    hard_spans.append([s, e])

            hard_spans = [sp for sp in hard_spans if (sp[1] - sp[0]) >= BEST_MIN_SPAN_LEN]
            hard_spans = merge_spans(hard_spans, gap=BEST_MERGE_GAP)
            out_rows.append({"id": rec["id"], "soft_labels": soft_labels, "hard_labels": hard_spans})

    write_jsonl(out_jsonl, out_rows)


clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", "mu-shroom")
KIT = os.path.abspath("mu-shroom/participant_kit")
scorer = load_scorer_module(KIT)
print("Using participant kit:", KIT)

In [ ]:
# EN-transfer only notebook: no language-specific fine-tuning helpers needed.
print("Skipping language-specific fine-tuning helpers (EN-transfer only mode).")

In [ ]:
# Run cross-lingual EN-transfer evaluation only
if not os.path.isdir(EN_MODEL_DIR):
    raise FileNotFoundError(f"Expected English model at {EN_MODEL_DIR}. Train/copy it first.")

rows = []
for lang in TARGET_LANGS:
    print(f"\n=== Language: {lang} ===")
    ds_lang = load_dataset("Helsinki-NLP/mu-shroom", lang)

    ref_jsonl = os.path.join(OUT_ROOT, f"reference_{lang}_validation.jsonl")
    write_jsonl(
        ref_jsonl,
        [
            {
                "id": row["id"],
                "model_output_text": row["model_output_text"],
                "soft_labels": row["soft_labels"],
                "hard_labels": row["hard_labels"],
            }
            for row in ds_lang["validation"]
        ],
    )

    pred_transfer = os.path.join(OUT_ROOT, f"pred_transfer_en_to_{lang}.jsonl")
    score_transfer = os.path.join(OUT_ROOT, f"scores_transfer_en_to_{lang}.txt")
    predict_with_params(EN_MODEL_DIR, ref_jsonl, pred_transfer)
    iou_t, cor_t = score_predictions(scorer, ref_jsonl, pred_transfer, score_transfer)
    rows.append({
        "Language": lang,
        "Setting": "EN-transfer",
        "IoU": iou_t,
        "Correlation": cor_t,
        "Prediction file": pred_transfer,
    })
    print(f"EN-transfer -> IoU: {iou_t:.6f}, Cor: {cor_t:.6f}")

results_df = pd.DataFrame(rows).sort_values(by=["Language"]).reset_index(drop=True)
print("\nPer-language official EN-transfer results:")
display(results_df)

In [ ]:
# EN-transfer-only summary table
if "results_df" not in globals() or results_df.empty:
    raise RuntimeError("Run the previous cell first.")

summary_df = (
    results_df[["Language", "IoU", "Correlation", "Prediction file"]]
    .sort_values(by="IoU", ascending=False)
    .reset_index(drop=True)
)

print("EN-transfer summary (ranked by IoU):")
display(summary_df)